# Comparative accuracy plot

Load all `results.json` files under `cl/results` and create a clean comparative `accuracy_per_epoch` plot. The figure has no title, uses a white background, and distinguishes experiments with colors, line styles, and markers.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

RESULTS_ROOT = Path.cwd() / 'results'
if not RESULTS_ROOT.is_dir():
    RESULTS_ROOT = Path.cwd() / 'cl' / 'results'
OUTPUT_PATH = RESULTS_ROOT / 'comparative_accuracy_per_epoch.png'
print(f'Results directory: {RESULTS_ROOT.resolve()}')


In [ ]:
result_paths = sorted(
    path
    for path in RESULTS_ROOT.glob('*/results.json')
    if '224x224' not in path.parent.name.lower()
)
if not result_paths:
    raise FileNotFoundError(
        f'No eligible results.json found under {RESULTS_ROOT}'
    )

results = {}
for path in result_paths:
    with path.open('r', encoding='utf-8') as file:
        results[path.parent.name] = json.load(file)

print(f'Loaded {len(results)} eligible experiments:')
for name in results:
    print(f' - {name}')

The next cell converts both result formats to one curve per experiment. For each recorded point, accuracy is averaged over the task values available at that point. Joint-training files use `accuracy_epochs`; sequential files use one point per task.

In [ ]:
def comparative_curve(result_path, result):
    plot_points_path = result_path.parent / 'plot_points.json'
    if plot_points_path.is_file():
        with plot_points_path.open('r', encoding='utf-8') as file:
            plot_points = json.load(file)
        accuracy_points = plot_points.get('accuracy_per_epoch')
        if accuracy_points is not None:
            x = np.asarray(accuracy_points['x'], dtype=float)
            values = np.asarray(accuracy_points['y'], dtype=float) / 100.0
            return x, values

    history = np.asarray(result.get('accuracy_history', []), dtype=float)
    if history.ndim != 2:
        raise ValueError(f'Unsupported accuracy_history format for {result_path}')
    num_tasks, epochs_per_task = history.shape
    values = []
    for task_id in range(num_tasks):
        start = task_id * epochs_per_task // num_tasks
        end = (task_id + 1) * epochs_per_task // num_tasks
        values.extend(history[task_id, start:end])
    values = np.asarray(values, dtype=float)
    x = np.arange(1, len(values) + 1, dtype=float)
    return x, values

curves = {
    name: comparative_curve(path, results[name])
    for name, path in [(path.parent.name, path) for path in result_paths]
}
for name, (x, values) in curves.items():
    print(f'{name}: {len(values)} exact points, x={x[0]:g}..{x[-1]:g}')

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, max(len(curves), 1)))
line_styles = ['-', '--', '-.', ':']
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']

figure, axis = plt.subplots(figsize=(12, 7), facecolor='white')
figure.patch.set_facecolor('white')
axis.set_facecolor('white')
for index, (name, (x, values)) in enumerate(curves.items()):
    axis.plot(
        x, values,
        color=colors[index % len(colors)],
        linestyle=line_styles[index % len(line_styles)],
        marker=markers[index % len(markers)],
        linewidth=2,
        markersize=5,
        markevery=max(1, len(values) // 12),
        label=name,
    )
axis.set_xlabel('Epoch / recorded point', color='black')
axis.set_ylabel('Accuracy', color='black')
axis.set_ylim(0, 1)
axis.tick_params(axis='both', colors='black')
for spine in axis.spines.values():
    spine.set_color('black')
axis.grid(True, alpha=0.25)
legend = axis.legend(loc='best', frameon=True, facecolor='white')
for text in legend.get_texts():
    text.set_color('black')
figure.tight_layout()
figure.savefig(OUTPUT_PATH, dpi=300, facecolor='white', bbox_inches='tight', pad_inches=0.05)
plt.show()
print(f'Saved comparative plot to: {OUTPUT_PATH.resolve()}')